<a href="https://colab.research.google.com/github/gangababupokanati/ai-mentor-portfolio/blob/main/Day9_StudyAgent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q langgraph langchain-google-genai langchain-community duckduckgo-search

import os, getpass
if 'GEMINI_API_KEY' not in os.environ:
    os.environ['GEMINI_API_KEY'] = getpass.getpass('Gemini API key: ')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 62.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 67.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
Gemini API key: ··········


In [3]:
!pip install -q -U ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 12.3 MB/s eta 0:00:00


In [4]:
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun

@tool
def web_search(query: str) -> str:
    """Search the web for up-to-date information.
    Use when the question requires current events, recent facts, or
    information not in static training knowledge."""
    return DuckDuckGoSearchRun().run(query)

# Test it works
print(web_search.invoke({'query': 'TCS hiring 2026'})[:400])

...Hiring 2026 is a flagship national-level recruitment program by Tata Consultancy Services (TCS) exclusively for science and computer application graduates from the 2025 and 2026... TCS MBA Hiring 2026: Tata Consultancy Services (TCS) has officially announced its MBA Hiring Drive for the Batch … Tata Consultancy Services (TCS), a global leader in IT services, consulting, and business solutions, 


In [5]:
from langgraph.prebuilt import create_react_agent
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash')
agent = create_react_agent(llm, tools=[web_search])

print('Agent created.')

Agent created.


/tmp/ipykernel_1668/1070485237.py:5: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools=[web_search])


In [6]:
result = agent.invoke({
    'messages': [('user', "What is TCS's 2026 hiring quota?")]
})

for i, m in enumerate(result['messages']):
    print(f'\n[{i}] {type(m).__name__}')
    if hasattr(m, 'content'):
        print(f'    Content: {str(m.content)[:300]}')
    if hasattr(m, 'tool_calls') and m.tool_calls:
        print(f'    Tool calls: {m.tool_calls}')


[0] HumanMessage
    Content: What is TCS's 2026 hiring quota?

[1] AIMessage
    Content: 
    Tool calls: [{'name': 'web_search', 'args': {'query': 'TCS 2026 hiring quota'}, 'id': 'fe5cabe7-eadb-4883-ac3a-bd6fcd257fa8', 'type': 'tool_call'}]

[2] ToolMessage
    Content: India's largest IT services firm, Tata Consultancy Services (TCS), has made 25,000 job offers to fresh graduates for FY2026-27, signaling a cautious yet continued commitment to campus hiring ... TCS Hiring 2026: 40 अरब डॉलर की डील! अब देश की सबसे बड़ी IT कंपनी में फ्रेशर्स के लिए 25 हजार नई नौकरियां

[3] AIMessage
    Content: [{'type': 'text', 'text': 'TCS has made 25,000 job offers to fresh graduates for FY2026-27.', 'extras': {'signature': 'CrgCAQw51sc+nHzJKTIIZqlG7ONLlQjAIBx+aquDcEx3cOpxS76WeodK3nclD4SMmI53P8wdkntmv/1h7pk8LDGXkzKtkL+fAy/bC2T/pUXGJ5LT+qTxFJSuFY9NRKsVTxBjd0NRnlO8sfqc7Bc7eMEGN7aCEdOoABNBNOl4dXNGj/kw8ytpv


## Day 9 Lab 9A — Trace as a story

1. **Human asked:** "What is TCS's 2026 hiring quota?"
2. **Agent thought:** "I don't know recent figures. I should search."
3. **Agent acted:** called `web_search('TCS 2026 hiring quota')`.
4. **Agent observed:** got back search results mentioning 40-50K range.
5. **Agent answered:** synthesised "Based on search results, TCS plans to hire 40-50K freshers..."

This is the ReAct loop. Every agent we build follows this pattern.

In [9]:
result = agent.invoke({
    'messages': [('user', 'Search this URL and tell me what it says: https://this-domain-does-not-exist-12345.example.com/jd')]
})

for i, m in enumerate(result['messages']):
    print(f'\n[{i}] {type(m).__name__}')
    if hasattr(m, 'content'):
        print(f'    {str(m.content)[:300]}')


[0] HumanMessage
    Search this URL and tell me what it says: https://this-domain-does-not-exist-12345.example.com/jd

[1] AIMessage
    [{'type': 'text', 'text': 'I cannot directly search a URL and tell you what it says. The `web_search` tool takes a search query, not a URL.', 'extras': {'signature': 'CosDAQw51seVVtnjQhKSqZCK6odwTFQwyG9WDf/4BsDzve7oGh0wnoZmOiE87l8ZMvpPxLHTjvE23hm+dIba6rZeO8Q+LLsxvdDCbJip8cTF4MzMLNcVwyoqhE/9xzDIbBelG
